In [31]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split,GridSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [32]:
df = pd.read_csv('adult.csv')

In [33]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [34]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education.num',
       'marital.status', 'occupation', 'relationship', 'race', 'sex',
       'capital.gain', 'capital.loss', 'hours.per.week', 'native.country',
       'income'],
      dtype='object')

In [35]:
df.shape

(32561, 15)

In [36]:
df.describe

<bound method NDFrame.describe of        age workclass  fnlwgt     education  education.num      marital.status  \
0       90         ?   77053       HS-grad              9             Widowed   
1       82   Private  132870       HS-grad              9             Widowed   
2       66         ?  186061  Some-college             10             Widowed   
3       54   Private  140359       7th-8th              4            Divorced   
4       41   Private  264663  Some-college             10           Separated   
...    ...       ...     ...           ...            ...                 ...   
32556   22   Private  310152  Some-college             10       Never-married   
32557   27   Private  257302    Assoc-acdm             12  Married-civ-spouse   
32558   40   Private  154374       HS-grad              9  Married-civ-spouse   
32559   58   Private  151910       HS-grad              9             Widowed   
32560   22   Private  201490       HS-grad              9       Never-marri

In [37]:
df.isna().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [38]:
df.duplicated().sum()

np.int64(24)

In [39]:
df[df.duplicated()]

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
8453,25,Private,308144,Bachelors,13,Never-married,Craft-repair,Not-in-family,White,Male,0,0,40,Mexico,<=50K
8645,90,Private,52386,Some-college,10,Never-married,Other-service,Not-in-family,Asian-Pac-Islander,Male,0,0,35,United-States,<=50K
12202,21,Private,250051,Some-college,10,Never-married,Prof-specialty,Own-child,White,Female,0,0,10,United-States,<=50K
14346,20,Private,107658,Some-college,10,Never-married,Tech-support,Not-in-family,White,Female,0,0,10,United-States,<=50K
15603,25,Private,195994,1st-4th,2,Never-married,Priv-house-serv,Not-in-family,White,Female,0,0,40,Guatemala,<=50K
17344,21,Private,243368,Preschool,1,Never-married,Farming-fishing,Not-in-family,White,Male,0,0,50,Mexico,<=50K
19067,46,Private,173243,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K
20388,30,Private,144593,HS-grad,9,Never-married,Other-service,Not-in-family,Black,Male,0,0,40,?,<=50K
20507,19,Private,97261,HS-grad,9,Never-married,Farming-fishing,Not-in-family,White,Male,0,0,40,United-States,<=50K
22783,19,Private,138153,Some-college,10,Never-married,Adm-clerical,Own-child,White,Female,0,0,10,United-States,<=50K


In [40]:
# df[df.duplicated(keep=False)]

In [41]:
df = df.drop_duplicates()

In [42]:
df.duplicated().sum()

np.int64(0)

In [43]:
df['workclass'].value_counts()

workclass
Private             22673
Self-emp-not-inc     2540
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

In [44]:
df['occupation'].value_counts()

occupation
Prof-specialty       4136
Craft-repair         4094
Exec-managerial      4065
Adm-clerical         3768
Sales                3650
Other-service        3291
Machine-op-inspct    2000
?                    1843
Transport-moving     1597
Handlers-cleaners    1369
Farming-fishing       992
Tech-support          927
Protective-serv       649
Priv-house-serv       147
Armed-Forces            9
Name: count, dtype: int64

In [45]:
df = df.replace('?',np.nan)

In [46]:
df.drop(columns = 'education.num',inplace=True)

In [47]:
df.isna().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     582
income               0
dtype: int64

In [48]:
df.isna().sum().sum()

np.int64(4261)

In [49]:
x = df.drop(columns = 'income')

In [50]:
y = y = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

In [51]:
obj_cols = x.select_dtypes(include = 'object').columns

In [52]:
x[obj_cols] = x[obj_cols].astype('category')

In [53]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,train_size=0.8,random_state=42)

In [54]:
xgboost_classifier = XGBClassifier(
    enable_categorical=True,
    random_state = 42
)

In [65]:
grid_search_cv = GridSearchCV(
                    estimator=xgboost_classifier,
                    param_grid = {
                            'n_estimators': [100, 200, 300],
                            'learning_rate': [0.01, 0.05, 0.1],
                            'max_depth': [3, 5, 7],
                            'min_child_weight': [1, 3, 5],
                            'subsample': [0.8, 1.0],
                            'colsample_bytree': [0.8, 1.0],
                            'gamma': [0, 0.1, 0.3],
                            'reg_alpha': [0, 0.1, 1],
                            'reg_lambda': [1, 3, 5],
                            'scale_pos_weight': [1, 2, 3]
                        },
                        n_jobs=-1,
                        verbose=10
)

In [66]:
grid_search_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 26244 candidates, totalling 131220 fits


KeyboardInterrupt: 

In [60]:
y_pred_train = grid_search_cv.predict(xtrain)
y_pred_test = grid_search_cv.predict(xtest)

In [61]:
print(classification_report(ytrain,y_pred_train))

              precision    recall  f1-score   support

           0       0.90      0.95      0.93     19710
           1       0.82      0.68      0.74      6319

    accuracy                           0.89     26029
   macro avg       0.86      0.81      0.83     26029
weighted avg       0.88      0.89      0.88     26029



In [62]:
print(classification_report(ytest,y_pred_test))

              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4988
           1       0.76      0.65      0.70      1520

    accuracy                           0.87      6508
   macro avg       0.83      0.79      0.81      6508
weighted avg       0.87      0.87      0.87      6508

